In [1]:
"""
Step 4a: Build event-synchronized data for the event-study design (Analysis 4).

For each of 11 named cybersecurity incidents (2018-2026), extracts a +/-30 day
window of daily burnout counts around the event date, so they can be pooled and
averaged across events regardless of when in the study period they occurred.

Reads daily_burnout_series.csv (built by build_daily_burnout_series.py -- NOT
the Granger q1_daily_series.csv, which is a different dataset built under a
different CVE condition and isn't used here at all).

For every metric column found in the daily file, produces two transformations:
  - deseasonalized residual (day-of-week + Patch-Tuesday pattern removed, same
    regression as your Granger pipeline)
  - a z-score normalized to each event's OWN pre-event baseline, so events from
    different years -- with very different absolute burnout volumes -- are
    comparable when pooled together. Without this, later, bigger-volume events
    would dominate a simple average and wash out earlier, smaller-scale ones.
"""

import pandas as pd
import numpy as np
import statsmodels.api as sm
import os

# ----------------------------------------------------------------------
# CONFIG
# ----------------------------------------------------------------------
OUT_DIR = "/Users/nadia/Desktop/redditRun_june/event_study_v2/"
DAILY_CSV = os.path.join(OUT_DIR, "daily_burnout_series.csv")

WINDOW = 30  # days before/after event

# Verified dates -- see conversation for sourcing. All are >2 months apart, so
# no window overlaps at WINDOW=30.
EVENTS = [
    ("Starwood/Marriott breach",  "2018-11-30"),
    ("Capital One breach",        "2019-07-29"),
    ("SolarWinds",                "2020-12-13"),
    ("Log4Shell",                 "2021-12-09"),
    ("Follina",                   "2022-05-30"),
    ("MOVEit",                    "2023-05-31"),
    ("Citrix Bleed",              "2023-10-10"),
    ("Snowflake breach",          "2024-05-23"),
    ("CrowdStrike outage",        "2024-07-19"),
    ("npm worm (Shai-Hulud)",     "2025-09-15"),
    ("Stryker cyberattack",       "2026-03-11"),
]


def is_patch_tuesday(date):
    return 1 if (date.weekday() == 1 and 8 <= date.day <= 14) else 0


def deseasonalize(daily_df, date_col, value_cols):
    """Same day-of-week + Patch-Tuesday regression used in the Granger script."""
    dates = pd.to_datetime(daily_df[date_col])
    dow = dates.dt.dayofweek
    patch_tues = dates.apply(is_patch_tuesday)

    dow_dummies = pd.get_dummies(dow, prefix="dow", drop_first=True)
    X = pd.concat([dow_dummies, patch_tues.rename("patch_tuesday")], axis=1)
    X = sm.add_constant(X).astype(float)

    residuals = {}
    for col in value_cols:
        y = daily_df[col].astype(float)
        model = sm.OLS(y, X).fit()
        residuals[col] = model.resid

    return pd.DataFrame(residuals)


def main():
    if not os.path.exists(DAILY_CSV):
        print(f"Could not find {DAILY_CSV}. Run build_daily_burnout_series.py first.")
        return

    daily = pd.read_csv(DAILY_CSV)
    daily["date"] = pd.to_datetime(daily["date"])
    print(f"Loaded {len(daily)} days from {DAILY_CSV}")

    metric_cols = [c for c in daily.columns if c not in ("date", "n_posts_total")]
    print(f"Metrics found: {metric_cols}")

    print("\nDeseasonalizing (day-of-week + Patch Tuesday)...")
    resid = deseasonalize(daily, "date", metric_cols)
    for col in metric_cols:
        daily[f"{col}_resid"] = resid[col]

    rows = []
    for event_name, event_date_str in EVENTS:
        event_date = pd.Timestamp(event_date_str)
        window_start = event_date - pd.Timedelta(days=WINDOW)
        window_end = event_date + pd.Timedelta(days=WINDOW)

        window_df = daily[(daily["date"] >= window_start) & (daily["date"] <= window_end)].copy()
        if window_df.empty:
            print(f"  WARNING: no data found for {event_name} ({event_date_str}) -- skipping "
                  f"(check that daily_burnout_series.csv covers this date range)")
            continue

        window_df["event_name"] = event_name
        window_df["event_date"] = event_date
        window_df["days_from_event"] = (window_df["date"] - event_date).dt.days

        pre_mask = window_df["days_from_event"] < 0
        n_pre_days = pre_mask.sum()
        if n_pre_days < WINDOW:
            print(f"  NOTE: {event_name} only has {n_pre_days}/{WINDOW} pre-event days "
                  f"available (likely near the start of your data) -- baseline may be noisy")

        for col in metric_cols:
            resid_col = f"{col}_resid"
            pre_mean_raw = window_df.loc[pre_mask, col].mean()
            pre_mean_resid = window_df.loc[pre_mask, resid_col].mean()
            pre_std_resid = window_df.loc[pre_mask, resid_col].std()

            window_df[f"{col}_norm"] = (
                window_df[col] / pre_mean_raw if pre_mean_raw > 0 else np.nan
            )
            window_df[f"{col}_resid_z"] = (
                (window_df[resid_col] - pre_mean_resid) / pre_std_resid
                if pre_std_resid and pre_std_resid > 0 else np.nan
            )

        rows.append(window_df)
        print(f"  {event_name} ({event_date_str}): {len(window_df)} days in window")

    if not rows:
        print("No events matched your data range -- nothing to save.")
        return

    aligned = pd.concat(rows, ignore_index=True)

    keep_cols = (
        ["event_name", "event_date", "date", "days_from_event"]
        + metric_cols
        + [f"{c}_resid" for c in metric_cols]
        + [f"{c}_norm" for c in metric_cols]
        + [f"{c}_resid_z" for c in metric_cols]
    )
    aligned = aligned[keep_cols]

    out_path = os.path.join(OUT_DIR, "event_study_aligned.csv")
    aligned.to_csv(out_path, index=False)
    print(f"\nSaved -> {out_path}")
    print(f"Total rows: {len(aligned)} across {aligned['event_name'].nunique()} events")


if __name__ == "__main__":
    main()

Loaded 3037 days from /Users/nadia/Desktop/redditRun_june/event_study_v2/daily_burnout_series.csv
Metrics found: ['n_burnout', 'n_EX', 'n_EMO', 'n_COG', 'n_MD']

Deseasonalizing (day-of-week + Patch Tuesday)...
  Starwood/Marriott breach (2018-11-30): 61 days in window
  Capital One breach (2019-07-29): 61 days in window
  SolarWinds (2020-12-13): 61 days in window
  Log4Shell (2021-12-09): 61 days in window
  Follina (2022-05-30): 61 days in window
  MOVEit (2023-05-31): 61 days in window
  Citrix Bleed (2023-10-10): 61 days in window
  Snowflake breach (2024-05-23): 61 days in window
  CrowdStrike outage (2024-07-19): 61 days in window
  npm worm (Shai-Hulud) (2025-09-15): 61 days in window
  Stryker cyberattack (2026-03-11): 61 days in window

Saved -> /Users/nadia/Desktop/redditRun_june/event_study_v2/event_study_aligned.csv
Total rows: 671 across 11 events
